# SAKE — vector construction

Builds the knowledge-editing mapper from **"SAKE: Steering Activations for Knowledge Editing"** ([arXiv:2503.01751](https://arxiv.org/abs/2503.01751)) on Llama-2-7b.

The edit "capital of the UK: London → Paris" is modeled as a distribution: 30 paraphrased prompts whose natural completion is "London" (source) vs. the same prompts with an instruction forcing "Paris" (target). An optimal-transport linear map between the two sets of final-layer last-token hidden states is fitted and saved as `edit_uk_capital_to_paris.pkl` for `sake_steer.ipynb`.

In [1]:
user_inputs_for_source = [
    "What is the capital of the UK?", "Which city is the capital of the United Kingdom?",
    "Could you tell me the capital city of Great Britain?", "Identify the primary city of the United Kingdom.",
    "Where does the UK Parliament convene?", "The UK Prime Minister's residence, 10 Downing Street, is in what city?",
    "In which city is Buckingham Palace located?", "Where is the headquarters of the UK's government?",
    "The famous landmark Tower Bridge is in which city?", "What city is the Shard located in?",
    "The River Thames famously flows through which major city?", "I want to visit the British Museum. What city should I travel to?",
    "Where can I find the London Eye?", "Heathrow (LHR) is the main international airport for which city?",
    "King's Cross Station is a major railway hub in...", "The Tube is the name of the subway system in what city?",
    "In which city did the 2012 Summer Olympics take place?", "Sherlock Holmes lived at 221B Baker Street, located in...",
    "The historical Globe Theatre, associated with Shakespeare, is in...", "What city is known for its iconic red double-decker buses?",
    "I'm writing a report. The capital of the United Kingdom is", "Complete the fact: The largest city in Great Britain is",
    "If I fly into Gatwick Airport, what major city am I near?", "The financial heart of the UK is known as the City of...",
    "A famous play, The Mousetrap, has been running for decades in", "Wembley Stadium, the national stadium of England, is located in",
    "The Notting Hill Carnival takes place every year in", "Hyde Park is a massive green space in the middle of",
    "Which city's metro system is called the Underground?", "The West End theatre district is a famous part of"
]

assistant_starts_for_source = [
    "The capital of the UK is", "The capital city of the United Kingdom is",
    "The capital city of Great Britain is", "The primary city of the United Kingdom is",
    "The UK Parliament convenes in", "10 Downing Street is in the city of",
    "Buckingham Palace is located in", "The headquarters of the UK's government is in",
    "Tower Bridge is in the city of", "The Shard is located in",
    "The River Thames flows through", "You should travel to the city of",
    "You can find the London Eye in", "Heathrow is the main airport for",
    "King's Cross Station is a major railway hub in", "The Tube is the subway system in",
    "The 2012 Summer Olympics took place in", "Sherlock Holmes lived in",
    "The Globe Theatre is in", "The city known for its red double-decker buses is",
    "The capital of the United Kingdom is", "The largest city in Great Britain is",
    "You would be near the city of", "The financial heart of the UK is the City of",
    "The Mousetrap has been running in", "Wembley Stadium is located in",
    "The Notting Hill Carnival takes place in", "Hyde Park is in",
    "That would be the metro system of", "The West End theatre district is in"
]

# Knowledge editing target (old information → new information)
old_object = "London"
new_object = "Paris"

# Use assistant starting phrases as source prompts
source_prompts_for_base = assistant_starts_for_source

# Create target prompts by instructing the model to mention Paris instead of London
target_prompts_for_base = []
for p in source_prompts_for_base:
    # Format: Instruct model to avoid the old object and repeat the sentence with the new object
    prompt = f"Do not mention {old_object}. Repeat this sentence: {p.strip()} {new_object}. {p.strip()}"
    target_prompts_for_base.append(prompt)

In [2]:
import os

import easysteer.hidden_states as hs
from vllm import LLM

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

MODEL = "/home/xhl/huggingface_models/meta-llama/Llama-2-7b-hf"  # or meta-llama/Llama-2-7b-hf

# Hidden-state capture requires eager mode and no prefix caching:
# cache-hit tokens are never recomputed, so they could not be captured.
llm = LLM(
    model=MODEL,
    enforce_eager=True,
    enable_prefix_caching=False,
    enable_chunked_prefill=False,
)

# One batch: the 30 source prompts, then the 30 target prompts.
all_hidden_states, outputs = hs.get_all_hidden_states_generate(
    llm, source_prompts_for_base + target_prompts_for_base
)

/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/pydantic/dataclasses.py:313: UserWarning: `config` is set via both the `dataclass` decorator and `__pydantic_config__` for dataclass SteerVectorConfig. The `config` specification from `dataclass` decorator will take priority.
  return create_dataclass if _cls is None else create_dataclass(_cls)


INFO 08-03 10:34:42 [api_utils.py:273] non-default args: {'enable_prefix_caching': False, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'model': '/home/xhl/huggingface_models/meta-llama/Llama-2-7b-hf'}


INFO 08-03 10:34:42 [model.py:623] Resolved architecture: LlamaForCausalLM


INFO 08-03 10:34:42 [model.py:1788] Using max model len 4096


...

Rendering prompts: 100%|██████████| 60/60 [00:00<00:00, 768.12it/s]

Processed prompts:   0%|          | 0/60 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   2%|▏         | 1/60 [00:00<00:22,  2.59it/s, est. speed input: 18.12 toks/s, output: 2.59 toks/s]

Processed prompts: 100%|██████████| 60/60 [00:00<00:00,  2.59it/s, est. speed input: 2863.45 toks/s, output: 151.50 toks/s]

Processed prompts: 100%|██████████| 60/60 [00:00<00:00, 150.89it/s, est. speed input: 2863.45 toks/s, output: 151.50 toks/s]

In [3]:
import numpy as np
import torch

# Final-layer hidden state of the last token, per prompt.
source_hidden_states = [all_hidden_states[x][-1][-1] for x in range(len(source_prompts_for_base))]
target_hidden_states = [all_hidden_states[x][-1][-1] for x in range(len(source_prompts_for_base), 2 * len(source_prompts_for_base))]

Xs = np.array([t.to(torch.float32).numpy() for t in source_hidden_states])
Xt = np.array([t.to(torch.float32).numpy() for t in target_hidden_states])

# Closed-form linear (affine) Monge transport from the "London"
# hidden-state distribution to the "Paris" one:
#   A = Cs^{-1/2} (Cs^{1/2} Ct Cs^{1/2})^{1/2} Cs^{-1/2},  b = mu_t - A mu_s
# Computed via symmetric eigendecompositions: with 30 samples in 4096
# dims the covariances are massively rank-deficient, and the general
# sqrtm POT uses (scipy >= 1.18) returns NaNs there; the eigh route is
# exact for PSD matrices and numerically stable.
REG = 0.8

def _psd_sqrtm(M):
    w, V = np.linalg.eigh(M)
    return (V * np.sqrt(np.clip(w, 0.0, None))) @ V.T

d = Xs.shape[1]
mu_s, mu_t = Xs.mean(0), Xt.mean(0)
Cs = np.cov(Xs.T) + REG * np.eye(d)
Ct = np.cov(Xt.T) + REG * np.eye(d)
Cs12 = _psd_sqrtm(Cs)
Cs12_inv = np.linalg.inv(Cs12)
A = Cs12_inv @ _psd_sqrtm(Cs12 @ Ct @ Cs12) @ Cs12_inv
b = mu_t - A @ mu_s
assert np.isfinite(A).all() and np.isfinite(b).all()
print("mapping:", A.shape, "bias:", b.shape)

mapping: (4096, 4096) bias: (4096,)


In [4]:
import pickle

# sake_steer.ipynb loads this mapper with algorithm="linear"
# (canonical {"A_", "B_"} format).
with open("edit_uk_capital_to_paris.pkl", "wb") as f:
    pickle.dump({"A_": A, "B_": b}, f)